# 06 · An AI agent that trades, checked by maths

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kolyamkl/qm-fintech-quant-workshop/blob/main/notebooks/06_ai_risk_agent.ipynb)

*QM FinTech Society quant workshop · Led by **Adnan Salehin** (Lead SWE & VP, JPMorgan)*

Can a language model manage a portfolio? This notebook lets one try, safely. The design rule:

> **The model proposes, the code decides.**

1. On a day a risk line is crossed, our code computes the numbers: drawdown, VaR, each asset's volatility and share of the risk.
2. The model sees **only those numbers**, and proposes new weights plus a short explanation.
3. Our code runs the **same `check_plan`** from notebook 05. A failing plan is sent back with the reasons; after 3 tries the portfolio simply holds.
4. Our code checks that **every number in the explanation** really came from our snapshot.
5. The backtest scores the agent like any other strategy.

Everything is simulated. Nothing here touches a real account.

**Two ways to run it:**
- **With a free Gemini key**: a real model makes the decisions. Get a key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey), then in Colab click the **key icon** on the left, add a secret named `GEMINI_API_KEY`, and allow this notebook to use it. **Never paste a key into a cell.**
- **Without a key**: the notebook uses an **offline stand-in that is not AI**. It answers with the rule-based plan in the same format, so every cell still runs and you can see the whole loop. Its results equal the rules from notebook 05.

> **Running in Google Colab?** If Colab warns that this notebook wasn't written by Google, click **Run anyway**. Then use **File → Save a copy in Drive** so your changes are kept.

## Setup

Run this cell first. In Google Colab it downloads the workshop code and price data from GitHub; on a laptop it simply finds them.

In [1]:
import os, subprocess, sys

REPO = "qm-fintech-quant-workshop"
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", f"https://github.com/kolyamkl/{REPO}.git"], check=True)
    os.chdir(REPO)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", 120)

In [2]:
from backtester import load_closes, risk
from backtester.agent import AgentManager, OfflineAgent, cached, gemini, snapshot, INSTRUCTIONS

API_KEY = os.environ.get("GEMINI_API_KEY", "")
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY") or API_KEY
except Exception:
    pass

if API_KEY:
    ask = cached(gemini(API_KEY), folder="data/agent_cache/gemini")
    def make_agent(**options):
        return AgentManager(ask=ask, **options)
    MODE = "Gemini (a real model)"
else:
    def make_agent(**options):
        return OfflineAgent(**options)
    MODE = "offline stand-in (NOT AI: the rule-based plan in the model's format)"
print("Running with:", MODE)

Running with: offline stand-in (NOT AI: the rule-based plan in the model's format)


Same portfolio, same calm assets and same limits as notebook 05.

In [3]:
TECH = ["AAPL", "MSFT", "AMZN", "GOOGL", "QQQ"]
CALM = ["TLT", "IEF", "GLD", "XLP", "XLU", "XLV"]
prices = load_closes(TECH + CALM, start="2019-06-01", end="2023-12-31").dropna()
start = {t: 0.2 for t in TECH}
limits = risk.Limits(max_drawdown=0.15, max_daily_loss=0.03)

## 1. What the model is allowed to see

On 11 March 2020, the day the World Health Organization called Covid a pandemic, the tech portfolio crossed a line. This is the snapshot our code computes for that day. **The model sees nothing else.**

In [4]:
DAY = "2020-03-11"
decision = risk.decide_on(prices, start, DAY, limits, calm=CALM)
print("Trigger:", decision["trigger"], "|", " ".join(decision["reasons"]))

context = decision["context"]
shown = snapshot(context)
{k: shown[k] for k in ["date", "trigger", "drawdown", "one_day_var", "weights", "risk_share", "calm_candidates", "limits"]}

Trigger: emergency | Drawdown -17.4% is past the emergency line of -12.0%.


{'date': '2020-03-11',
 'trigger': 'emergency',
 'drawdown': -0.1741,
 'one_day_var': 0.0501,
 'weights': {'AAPL': 0.2, 'MSFT': 0.2, 'AMZN': 0.2, 'GOOGL': 0.2, 'QQQ': 0.2},
 'risk_share': {'AAPL': 0.2385,
  'MSFT': 0.2244,
  'AMZN': 0.1659,
  'GOOGL': 0.1837,
  'QQQ': 0.1875},
 'calm_candidates': ['TLT', 'IEF', 'GLD', 'XLP', 'XLU', 'XLV'],
 'limits': {'var_budget': 0.03, 'rotate_at': 0.075, 'emergency_at': 0.12}}

And these are the instructions it gets. Read them: they are the whole "agent".

In [5]:
print(INSTRUCTIONS.format(date=DAY, snapshot="{ ...the snapshot above... }"))

You are a risk manager for a student's paper portfolio. Nothing you say trades real money.
The investor has set limits on how far the portfolio may fall. A warning line was crossed today.

Your job: propose new target weights that bring the one-day VaR within the budget, by ROTATING weight
from the holdings with the largest risk_share into the calm_candidates, not by selling everything.

Rules:
- Use only tickers listed in "allowed". Weights are fractions of the portfolio, in steps of 0.05,
  none above 0.40, adding up to at most 1. Whatever is left over is cash.
- Use only the numbers in the snapshot. Do not use anything you know about what happened after 2020-03-11.
- Prefer the smallest change that fixes the problem.
- If you think no change is needed, return the current weights.

Answer with JSON only, in exactly this shape:
{"weights": {"TICKER": 0.20, "...": 0.10}, "explanation": "under 80 words, plain English, quoting numbers from the snapshot"}

Snapshot:
{ ...the snapshot abov

## 2. Ask once, and check the answer

In [6]:
agent = make_agent()
plan = agent(context)
entry = agent.log[-1]
for number, attempt in enumerate(entry["attempts"], 1):
    print(f"Attempt {number}:", attempt)
print("\nFinal plan:", plan)
print("Explanation:", entry["explanation"])
print("Numbers the model stated that our code never gave it:", entry["unsupported_numbers"] or "none")

Attempt 1: {'weights': {'AAPL': 0.05, 'MSFT': 0.05, 'AMZN': 0.1, 'GOOGL': 0.1, 'QQQ': 0.1, 'IEF': 0.4, 'GLD': 0.2}, 'passed': True, 'reasons': []}

Final plan: {'AAPL': 0.05, 'MSFT': 0.05, 'AMZN': 0.1, 'GOOGL': 0.1, 'QQQ': 0.1, 'IEF': 0.4, 'GLD': 0.2}
Explanation: One-day VaR is 5.01% against a budget of 3.00%. Moving weight from MSFT, AAPL, QQQ, AMZN, GOOGL into IEF, GLD.
Numbers the model stated that our code never gave it: none


## 3. The gate works even when the model does not

Here is a "model" that ignores the rules and goes all in. Watch the code reject it three times and hold.

In [7]:
import json

def reckless(prompt):
    return json.dumps({"weights": {"AAPL": 0.9, "MSFT": 0.1}, "explanation": "Buy the dip! Apple will bounce 30%."})

stubborn = AgentManager(ask=reckless)
print("Plan:", stubborn(context))
for attempt in stubborn.log[-1]["attempts"]:
    print(attempt["reasons"])

Plan: None
['Largest holding is 90%, above the 40% cap.']
['Largest holding is 90%, above the 40% cap.']
['Largest holding is 90%, above the 40% cap.']


That is the most important cell in this notebook. However clever or confused the model is, **it cannot trade outside the rules**, because the rules are plain Python that runs after it.

## 4. Replay 2019 to 2023 with the agent

The model is only asked on days a line is crossed (at most once every 5 days), so the whole replay needs a few dozen calls, not a thousand. With a key the first run takes a few minutes (calls are spaced to respect the free tier); every reply is **cached** in `data/agent_cache/`, so a rerun is instant and gives the same answers.

In [8]:
hold = risk.run_guarded(prices, start, limits, manager=None, calm=CALM, name="Buy & hold")
rules = risk.run_guarded(prices, start, limits, manager=risk.rotate_manager, calm=CALM, name="Rotate (rules)")
agent = make_agent()
with_agent = risk.run_guarded(prices, start, limits, manager=agent, calm=CALM, name="Rotate (agent)")
risk.compare([hold, rules, with_agent])

,Buy & hold,Rotate (rules),Rotate (agent)
Total return,163.3%,97.2%,97.2%
Annual return (CAGR),25.0%,16.9%,16.9%
Annual volatility,28.3%,17.4%,17.4%
Sharpe ratio,0.93,0.99,0.99
Max drawdown,-37.2%,-28.3%,-28.3%
Number of trades,0,77,77
Days over daily limit,45,12,12
Worst day,-11.5%,-5.8%,-5.8%
Costs paid,0.0%,0.6%,0.6%


In [9]:
decisions = agent.decisions()
print(f"Asked {len(decisions)} times; plans that passed: {(decisions.outcome == 'plan passed').sum()}; "
      f"explanations with made-up numbers: {(decisions['unsupported numbers'] != '').sum()}")
decisions.head(10)

Asked 24 times; plans that passed: 24; explanations with made-up numbers: 0


,date,trigger,attempts,outcome,explanation,unsupported numbers
0,2019-08-27,rotate,1,plan passed,"One-day VaR is 3.07% against a budget of 3.00%. Moving weight from MSFT, AAPL, GOOGL into IEF.",
1,2020-02-24,rotate,1,plan passed,One-day VaR is 2.52% against a budget of 3.00%. Keeping the current weights.,
2,2020-03-03,rotate,1,plan passed,"One-day VaR is 3.72% against a budget of 3.00%. Moving weight from AAPL, MSFT, GOOGL into IEF.",
3,2020-03-11,emergency,1,plan passed,"One-day VaR is 3.43% against a budget of 3.00%. Moving weight from AMZN, QQQ, GOOGL, AAPL, MSFT into GLD, IEF.",
4,2020-04-06,rotate,1,plan passed,"One-day VaR is 3.20% against a budget of 3.00%. Moving weight from QQQ, AAPL into IEF, GLD.",
5,2020-09-03,rotate,1,plan passed,One-day VaR is 2.75% against a budget of 3.00%. Moving weight from AAPL into IEF.,
6,2020-09-17,rotate,1,plan passed,One-day VaR is 2.57% against a budget of 3.00%. Keeping the current weights.,
7,2020-09-25,rotate,1,plan passed,One-day VaR is 2.76% against a budget of 3.00%. Moving weight from MSFT into IEF.,
8,2020-10-06,rotate,1,plan passed,One-day VaR is 2.57% against a budget of 3.00%. Keeping the current weights.,
9,2020-10-28,rotate,1,plan passed,One-day VaR is 2.63% against a budget of 3.00%. Keeping the current weights.,


**Read it like a risk manager:** did the agent beat the rules after costs? Did it trade more? Did any explanation quote a number our code never gave it? A model that invents a figure once will do it again; that is why we check every one.

## 5. The trap: an AI that already knows the future

A language model has read the news about 2020 and 2022. Told the date is 11 March 2020, it may "remember" the crash and the rebound, which no real investor knew that day. Its backtest can then look brilliant for the wrong reason: **look-ahead bias, AI edition**.

The test: run the agent again with the **dates hidden and tickers renamed** (`ASSET_A`, `ASSET_B`, ...). Same numbers, no memory to lean on. If performance drops a lot, the first run was partly cheating.

In [10]:
masked_agent = make_agent(anonymous=True)
masked = risk.run_guarded(prices, start, limits, manager=masked_agent, calm=CALM, name="Agent, names hidden")
risk.compare([with_agent, masked])

,Rotate (agent),"Agent, names hidden"
Total return,97.2%,97.2%
Annual return (CAGR),16.9%,16.9%
Annual volatility,17.4%,17.4%
Sharpe ratio,0.99,0.99
Max drawdown,-28.3%,-28.3%
Number of trades,77,77
Days over daily limit,12,12
Worst day,-5.8%,-5.8%
Costs paid,0.6%,0.6%


In [11]:
print(masked_agent.log[0]["explanation"] if masked_agent.log else "No decisions.")

One-day VaR is 3.07% against a budget of 3.00%. Moving weight from ASSET_A, ASSET_B, ASSET_D into ASSET_G.


With the offline stand-in the two columns are identical: it has no memory to cheat with. With a real model, compare them carefully. A fairer test still: replay only dates **after the model's training data ends**, which is the only truly unseen history.

## 6. ✏️ Your turn

- **Change the instructions.** Copy `INSTRUCTIONS`, ask for something different ("prefer gold over bonds", "never hold more than 2 tech funds"), and pass it in by editing `backtester/agent.py` or by writing your own `ask` that adds a line to the prompt. Does the gate still hold?
- **Try another model.** Any function `ask(prompt) -> reply text` works: Gemini, Claude, ChatGPT or a local model.
- **Make it cheaper.** Only ask the model when `risk.unusual_moves` shows a z-score below -3; use the rules otherwise.

In [12]:
def ask_with_extra_rule(prompt):
    extra = "\nExtra rule from the investor: prefer GLD over bonds when you rotate.\n"
    return agent.ask(prompt + extra)

if API_KEY:
    custom = AgentManager(ask=ask_with_extra_rule)
    custom_run = risk.run_guarded(prices, start, limits, manager=custom, calm=CALM, name="Agent + my rule")
    display(risk.compare([with_agent, custom_run]))
else:
    print("Add a GEMINI_API_KEY secret to try a real model with your own rule.")

Add a GEMINI_API_KEY secret to try a real model with your own rule.


**Questions to discuss:**
- Who is responsible when an AI agent loses money: the model, the person who wrote the rules, or the investor?
- What should never be left to the model, however good it gets? (Hint: look at what `check_plan` enforces.)
- An AI agent that explains itself in perfect English can still be wrong. How would you audit it?

> Educational use only. Simulated trades on historical data; not financial advice.